In [ ]:
# Install GoCommands (Linux x86_64)
GOCMD_VER=$(curl -L -s https://raw.githubusercontent.com/cyverse/gocommands/main/VERSION.txt); \
curl -L -s https://github.com/cyverse/gocommands/releases/download/${GOCMD_VER}/gocmd-${GOCMD_VER}-linux-amd64.tar.gz | tar zxvf -

# Configure iRODS (accept defaults for Host/Port/Zone; use your CyVerse username)
./gocmd init

# Quick sanity check: can you list your home?
./gocmd ls 

In [1]:
./gocmd get --progress -r "/iplant/home/shared/earthlab/macrosystems/field-data/output/summer-2023-10cm-10k"

SyntaxError: invalid syntax (137666420.py, line 1)

In [ ]:
spectralbridge/summer-2023-10cm-10k/SPR2-06-28-23-ExportPackage/NEON_D13_NIWO_test_aligned_orthomosaic.h5

In [ ]:
from __future__ import annotations

from pathlib import Path
import shutil
from datetime import datetime
import re
import h5py

# --- your inputs ---
h5_path = "summer-2023-10cm-10k/SPR2-06-28-23-ExportPackage/NEON_D13_NIWO_test_aligned_orthomosaic.h5"
base_out = "drone_outputs"


def _infer_yyyymmdd_from_path(p: Path) -> str:
    """
    Try to infer a date from the path like '.../SPR2-06-28-23-ExportPackage/...'
    Returns YYYYMMDD. Falls back to today's date if not found.
    """
    s = str(p)
    m = re.search(r"(\d{2})-(\d{2})-(\d{2})", s)  # MM-DD-YY
    if m:
        mm, dd, yy = m.group(1), m.group(2), m.group(3)
        return f"20{yy}{mm}{dd}"
    return datetime.now().strftime("%Y%m%d")


def find_reflectance_dataset_path(h5: h5py.File) -> str:
    preferred = [
        "NIWO/Reflectance/Reflectance_Data",
        "Reflectance/Reflectance_Data",
    ]
    for p in preferred:
        if p in h5 and isinstance(h5[p], h5py.Dataset):
            return p

    candidates = []
    def visitor(name, obj):
        if isinstance(obj, h5py.Dataset):
            lname = name.lower()
            if "reflect" in lname:
                score = 0
                if "reflectance_data" in lname:
                    score += 100
                if "reflectance" in lname:
                    score += 20
                score += 5 * max(0, obj.ndim - 1)
                try:
                    if obj.size and obj.size > 1_000_000:
                        score += 10
                except Exception:
                    pass
                candidates.append((score, name))
    h5.visititems(visitor)
    if not candidates:
        raise RuntimeError("Could not find a reflectance-like dataset in the H5.")
    candidates.sort(reverse=True)
    return candidates[0][1]


def patch_nodata_attrs(h5_file: Path, nodata_value: float = -9999.0) -> str:
    with h5py.File(h5_file, "r+") as h5:
        ds_path = find_reflectance_dataset_path(h5)
        ds = h5[ds_path]
        for key in ["_FillValue", "NoDataValue", "nodata", "no_data", "missing_value", "fill_value"]:
            if key not in ds.attrs:
                ds.attrs[key] = nodata_value
        return ds_path


def run_drone_h5_to_clean_output(
    src_h5: str | Path,
    base_out: str | Path,
    *,
    # These control the synthetic NEON name:
    domain: str = "D13",
    site: str = "NIWO",
    tile: str = "L000-0",
    yyyymmdd: str | None = None,  # inferred from path if None
    neon_kind: str = "directional_reflectance",  # keep NEON-ish
    # pipeline knobs
    product_code: str = "DP1.30006.001",
    resample_method: str | None = "convolution",
    brightness_offset: float | None = None,
    parquet_chunk_size: int = 50_000,
    nodata_value: float = -9999.0,
) -> dict[str, Path]:
    """
    Create a unique run root, copy the H5 into it using a NEON-parseable filename,
    patch nodata attrs in the run copy, then run process_one_flightline().
    """
    from spectralbridge.utils.naming import get_flight_paths
    from spectralbridge.pipelines.pipeline import process_one_flightline

    src_h5 = Path(src_h5).expanduser().resolve()
    base_out = Path(base_out).expanduser().resolve()
    if not src_h5.exists():
        raise FileNotFoundError(f"H5 not found: {src_h5}")

    # Unique run folder (so reruns don’t collide)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_root = base_out / f"run_{stamp}"
    run_root.mkdir(parents=True, exist_ok=True)

    # Build a NEON-parseable flight stem (no extra suffixes!)
    date_str = yyyymmdd or _infer_yyyymmdd_from_path(src_h5)
    flight_stem = f"NEON_{domain}_{site}_DP1_{tile}_{date_str}_{neon_kind}"

    # Canonical H5 destination uses flight_stem + ".h5"
    flight_paths = get_flight_paths(base_folder=run_root, flight_stem=flight_stem)
    h5_dest = Path(flight_paths["h5_path"])
    h5_dest.parent.mkdir(parents=True, exist_ok=True)

    shutil.copy2(src_h5, h5_dest)

    ds_path = patch_nodata_attrs(h5_dest, nodata_value=nodata_value)
    print(f"Patched nodata attrs on dataset: {ds_path}")
    print(f"Run root: {run_root}")
    print(f"H5 run copy: {h5_dest}")
    print(f"flight_stem: {flight_stem}")

    process_one_flightline(
        base_folder=run_root,
        product_code=product_code,
        flight_stem=flight_stem,
        resample_method=resample_method,
        brightness_offset=brightness_offset,
        parallel_mode=False,
        parquet_chunk_size=parquet_chunk_size,
        ray_cpus=None,
    )

    return {
        "run_root": run_root,
        "flight_stem": Path(flight_stem),
        "h5_source": src_h5,
        "h5_run_copy": h5_dest,
    }


# ---- run it ----
result = run_drone_h5_to_clean_output(
    src_h5=h5_path,
    base_out=base_out,
    # optional override if you want:
    # yyyymmdd="20230628",
    resample_method="convolution",
    nodata_value=-9999.0,
)

print("Done. Key paths:")
for k, v in result.items():
    print(f"{k}: {v}")


Patched nodata attrs on dataset: NIWO/Reflectance/Reflectance_Data
Run root: /home/jovyan/data-store/spectralbridge/drone_outputs/run_20260131_184518
H5 run copy: /home/jovyan/data-store/spectralbridge/drone_outputs/run_20260131_184518/NEON_D13_NIWO_DP1_L000-0_20230628_directional_reflectance.h5
flight_stem: NEON_D13_NIWO_DP1_L000-0_20230628_directional_reflectance


✅ All parquet files look consistent.
[merge] Start flightline=NEON_D13_NIWO_DP1_L000-0_20230628_directional_reflectance prefix=NEON_D13_NIWO_DP1_L000-0_20230628_directional_reflectance_brdfandtopo_corrected
[merge] Output parquet → /home/jovyan/data-store/spectralbridge/drone_outputs/run_20260131_184518/NEON_D13_NIWO_DP1_L000-0_20230628_directional_reflectance/NEON_D13_NIWO_DP1_L000-0_20230628_directional_reflectance_brdfandtopo_corrected_merged_pixel_extraction.parquet
[merge] Engine=duckdb memory_limit=64.0GB threads=4 row_group_size=auto temp_dir=/home/jovyan/data-store/spectralbridge/drone_outputs/run_20260131_184518/NEON_D13_NIWO_DP1_L000-0_20230628_directional_reflectance/.duckdb_tmp
[merge] Set DuckDB memory_limit = 64.0GB
[merge] Disabled insertion-order preservation to reduce memory usage
[merge] 🔍 Starting streaming merge for NEON_D13_NIWO_DP1_L000-0_20230628_directional_reflectance
[merge] 📊 Discovered inputs: orig=1, corr=1, resamp=14
[merge] 🔨 Building CTEs (streaming, no 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[merge]    ✅ Test query successful - returned 1000 rows
[merge]    ✅ Test query: no duplicate pixel_ids in sample
[merge]    Expected rows: 2,240,928
[merge]    Executing streaming COPY (this may take 5-15 minutes for large datasets)...
[merge]    You can check progress by monitoring file size: ls -lh NEON_D13_NIWO_DP1_L000-0_20230628_directional_reflectance_brdfandtopo_corrected_merged_pixel_extraction.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[merge]    ✅ Streaming COPY complete in 13.8 seconds (0.2 minutes)
[merge]    ✅ Output file created: 0.69 GB
[merge] ✅ Wrote parquet: /home/jovyan/data-store/spectralbridge/drone_outputs/run_20260131_184518/NEON_D13_NIWO_DP1_L000-0_20230628_directional_reflectance/NEON_D13_NIWO_DP1_L000-0_20230628_directional_reflectance_brdfandtopo_corrected_merged_pixel_extraction.parquet (exists=True)
[merge] 🔍 About to filter no-data rows...
[merge] 🔍 Filtering rows with >90% invalid spectral values (excluding raw_* columns)...
[merge]    Found 100 spectral columns to check
[merge]    Reading parquet file for filtering (2,240,928 rows)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[merge]    Checking 100 spectral columns for rows with majority invalid values (>90%)...


In [ ]:
from __future__ import annotations

from pathlib import Path
import shutil
from datetime import datetime
import re
import h5py

# --- batch inputs (we are in the spectralbridge folder, so repo-root relative paths) ---
H5_PATHS = [
    "summer-2023-10cm-10k/AOP-GOLDHILL-08-14-23-ExportPackage/NEON_D13_NIWO_test_aligned_orthomosaic.h5",
    "summer-2023-10cm-10k/AOP-GORDON-08-14-23-ExportPackage/NEON_D13_NIWO_test_aligned_orthomosaic.h5",
    "summer-2023-10cm-10k/AOP-MRS1-08-14-23-ExportPackage/NEON_D13_NIWO_test_aligned_orthomosaic.h5",
    "summer-2023-10cm-10k/AOP-MRS2-08-14-23-ExportPackage/NEON_D13_NIWO_test_aligned_orthomosaic.h5",
    "summer-2023-10cm-10k/AOP-Ruby-08-14-23-ExportPackage/NEON_D13_NIWO_test_aligned_orthomosaic.h5",
    "summer-2023-10cm-10k/CW1-08-08-23-ExportPackage/NEON_D13_NIWO_test_aligned_orthomosaic.h5",
    "summer-2023-10cm-10k/CW2-08-16-23-ExportPackage/NEON_D13_NIWO_test_aligned_orthomosaic.h5",
    "summer-2023-10cm-10k/CW3-08-16-23-ExportPackage/NEON_D13_NIWO_test_aligned_orthomosaic.h5",
    "summer-2023-10cm-10k/GAH1-07-25-23-ExportPackage/NEON_D13_NIWO_test_aligned_orthomosaic.h5",
    "summer-2023-10cm-10k/GAH2-07-25-23-ExportPackage/NEON_D13_NIWO_test_aligned_orthomosaic.h5",
    "summer-2023-10cm-10k/JC1-07-11-23-ExportPackage/NEON_D13_NIWO_test_aligned_orthomosaic.h5",
    "summer-2023-10cm-10k/OR3-08-16-23-ExportPackage/NEON_D13_NIWO_test_aligned_orthomosaic.h5",
    "summer-2023-10cm-10k/SH67W1-07-11-23-ExportPackage/NEON_D13_NIWO_test_aligned_orthomosaic.h5",
    "summer-2023-10cm-10k/SH67W2-07-11-23-ExportPackage/NEON_D13_NIWO_test_aligned_orthomosaic.h5",
    "summer-2023-10cm-10k/SH67_1-07-07-23-ExportPackage/NEON_D13_NIWO_test_aligned_orthomosaic.h5",
    "summer-2023-10cm-10k/SPR1-06-28-23-ExportPackage/NEON_D13_NIWO_test_aligned_orthomosaic.h5",
    "summer-2023-10cm-10k/SPR2-06-28-23-ExportPackage/NEON_D13_NIWO_test_aligned_orthomosaic.h5",
]

# Pick an output root (this folder will contain per-run folders)
base_out = "drone_outputs"


def _infer_yyyymmdd_from_path(p: Path) -> str:
    """
    Try to infer a date from the path like '.../SPR2-06-28-23-ExportPackage/...'
    Returns YYYYMMDD. Falls back to today's date if not found.
    """
    s = str(p)
    m = re.search(r"(\d{2})-(\d{2})-(\d{2})", s)  # MM-DD-YY
    if m:
        mm, dd, yy = m.group(1), m.group(2), m.group(3)
        return f"20{yy}{mm}{dd}"
    return datetime.now().strftime("%Y%m%d")


def find_reflectance_dataset_path(h5: h5py.File) -> str:
    preferred = [
        "NIWO/Reflectance/Reflectance_Data",
        "Reflectance/Reflectance_Data",
    ]
    for p in preferred:
        if p in h5 and isinstance(h5[p], h5py.Dataset):
            return p

    candidates = []
    def visitor(name, obj):
        if isinstance(obj, h5py.Dataset):
            lname = name.lower()
            if "reflect" in lname:
                score = 0
                if "reflectance_data" in lname:
                    score += 100
                if "reflectance" in lname:
                    score += 20
                score += 5 * max(0, obj.ndim - 1)
                try:
                    if obj.size and obj.size > 1_000_000:
                        score += 10
                except Exception:
                    pass
                candidates.append((score, name))
    h5.visititems(visitor)
    if not candidates:
        raise RuntimeError("Could not find a reflectance-like dataset in the H5.")
    candidates.sort(reverse=True)
    return candidates[0][1]


def patch_nodata_attrs(h5_file: Path, nodata_value: float = -9999.0) -> str:
    with h5py.File(h5_file, "r+") as h5:
        ds_path = find_reflectance_dataset_path(h5)
        ds = h5[ds_path]
        for key in ["_FillValue", "NoDataValue", "nodata", "no_data", "missing_value", "fill_value"]:
            if key not in ds.attrs:
                ds.attrs[key] = nodata_value
        return ds_path


def run_drone_h5_to_clean_output(
    src_h5: str | Path,
    base_out: str | Path,
    *,
    # These control the synthetic NEON name:
    domain: str = "D13",
    site: str = "NIWO",
    tile: str = "L000-0",
    yyyymmdd: str | None = None,  # inferred from path if None
    neon_kind: str = "directional_reflectance",  # keep NEON-ish
    # pipeline knobs
    product_code: str = "DP1.30006.001",
    resample_method: str | None = "convolution",
    brightness_offset: float | None = None,
    parquet_chunk_size: int = 50_000,
    nodata_value: float = -9999.0,
) -> dict[str, Path]:
    """
    Create a unique run root, copy the H5 into it using a NEON-parseable filename,
    patch nodata attrs in the run copy, then run process_one_flightline().
    """
    from spectralbridge.utils.naming import get_flight_paths
    from spectralbridge.pipelines.pipeline import process_one_flightline

    src_h5 = Path(src_h5).expanduser().resolve()
    base_out = Path(base_out).expanduser().resolve()
    if not src_h5.exists():
        raise FileNotFoundError(f"H5 not found: {src_h5}")

    # Unique run folder (so reruns don’t collide)
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    run_root = base_out / f"run_{stamp}"
    run_root.mkdir(parents=True, exist_ok=True)

    # Build a NEON-parseable flight stem (no extra suffixes!)
    date_str = yyyymmdd or _infer_yyyymmdd_from_path(src_h5)
    flight_stem = f"NEON_{domain}_{site}_DP1_{tile}_{date_str}_{neon_kind}"

    # Canonical H5 destination uses flight_stem + ".h5"
    flight_paths = get_flight_paths(base_folder=run_root, flight_stem=flight_stem)
    h5_dest = Path(flight_paths["h5_path"])
    h5_dest.parent.mkdir(parents=True, exist_ok=True)

    shutil.copy2(src_h5, h5_dest)

    ds_path = patch_nodata_attrs(h5_dest, nodata_value=nodata_value)
    print(f"Patched nodata attrs on dataset: {ds_path}")
    print(f"Run root: {run_root}")
    print(f"H5 run copy: {h5_dest}")
    print(f"flight_stem: {flight_stem}")

    process_one_flightline(
        base_folder=run_root,
        product_code=product_code,
        flight_stem=flight_stem,
        resample_method=resample_method,
        brightness_offset=brightness_offset,
        parallel_mode=False,
        parquet_chunk_size=parquet_chunk_size,
        ray_cpus=None,
    )

    return {
        "run_root": run_root,
        "flight_stem": Path(flight_stem),
        "h5_source": src_h5,
        "h5_run_copy": h5_dest,
    }


# -------------------------
# Batch loop over the files
# -------------------------
successes: list[dict[str, Path]] = []
failures: list[tuple[Path, str]] = []

print(f"Batch size: {len(H5_PATHS)}")
print(f"Base output: {Path(base_out).resolve()}")

for i, rel_path in enumerate(H5_PATHS, start=1):
    src = Path(rel_path)

    print(f"\n=== [{i}/{len(H5_PATHS)}] {src} ===")
    try:
        result = run_drone_h5_to_clean_output(
            src_h5=src,
            base_out=base_out,
            resample_method="convolution",
            nodata_value=-9999.0,
        )
        successes.append(result)

    except Exception as exc:
        msg = f"{type(exc).__name__}: {exc}"
        print(f"❌ FAILED: {msg}")
        failures.append((src, msg))

print("\n================================")
print("Batch complete")
print(f"  Successes: {len(successes)}")
print(f"  Failures:  {len(failures)}")

if failures:
    print("\nFailures:")
    for p, msg in failures:
        print(f" - {p}: {msg}")


Batch size: 17
Base output: /home/jovyan/data-store/spectralbridge/drone_outputs

=== [1/17] summer-2023-10cm-10k/AOP-GOLDHILL-08-14-23-ExportPackage/NEON_D13_NIWO_test_aligned_orthomosaic.h5 ===
Patched nodata attrs on dataset: NIWO/Reflectance/Reflectance_Data
Run root: /home/jovyan/data-store/spectralbridge/drone_outputs/run_20260131_201211
H5 run copy: /home/jovyan/data-store/spectralbridge/drone_outputs/run_20260131_201211/NEON_D13_NIWO_DP1_L000-0_20230814_directional_reflectance.h5
flight_stem: NEON_D13_NIWO_DP1_L000-0_20230814_directional_reflectance


✅ All parquet files look consistent.
[merge] Start flightline=NEON_D13_NIWO_DP1_L000-0_20230814_directional_reflectance prefix=NEON_D13_NIWO_DP1_L000-0_20230814_directional_reflectance_brdfandtopo_corrected
[merge] Output parquet → /home/jovyan/data-store/spectralbridge/drone_outputs/run_20260131_201211/NEON_D13_NIWO_DP1_L000-0_20230814_directional_reflectance/NEON_D13_NIWO_DP1_L000-0_20230814_directional_reflectance_brdfandtopo_corrected_merged_pixel_extraction.parquet
[merge] Engine=duckdb memory_limit=64.0GB threads=4 row_group_size=auto temp_dir=/home/jovyan/data-store/spectralbridge/drone_outputs/run_20260131_201211/NEON_D13_NIWO_DP1_L000-0_20230814_directional_reflectance/.duckdb_tmp
[merge] Set DuckDB memory_limit = 64.0GB
[merge] Disabled insertion-order preservation to reduce memory usage
[merge] 🔍 Starting streaming merge for NEON_D13_NIWO_DP1_L000-0_20230814_directional_reflectance
[merge] 📊 Discovered inputs: orig=1, corr=1, resamp=14
[merge] 🔨 Building CTEs (streaming, no 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[merge]    ✅ Test query successful - returned 1000 rows
[merge]    ✅ Test query: no duplicate pixel_ids in sample
[merge]    Expected rows: 7,068,376
[merge]    Executing streaming COPY (this may take 5-15 minutes for large datasets)...
[merge]    You can check progress by monitoring file size: ls -lh NEON_D13_NIWO_DP1_L000-0_20230814_directional_reflectance_brdfandtopo_corrected_merged_pixel_extraction.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[merge]    ✅ Streaming COPY complete in 35.0 seconds (0.6 minutes)
[merge]    ✅ Output file created: 2.27 GB
[merge] ✅ Wrote parquet: /home/jovyan/data-store/spectralbridge/drone_outputs/run_20260131_201211/NEON_D13_NIWO_DP1_L000-0_20230814_directional_reflectance/NEON_D13_NIWO_DP1_L000-0_20230814_directional_reflectance_brdfandtopo_corrected_merged_pixel_extraction.parquet (exists=True)
[merge] 🔍 About to filter no-data rows...
[merge] 🔍 Filtering rows with >90% invalid spectral values (excluding raw_* columns)...
[merge]    Found 100 spectral columns to check
[merge]    Reading parquet file for filtering (7,068,376 rows)...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[merge]    Checking 100 spectral columns for rows with majority invalid values (>90%)...
